In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


def find_project_root(
    marker: str = "pyproject.toml",
) -> Path:

    start = Path.cwd().resolve()

    for candidate in (start, *start.parents):
        if (candidate / marker).exists():
            return candidate

    raise FileNotFoundError(
        f"Project root not found. Missing marker: {marker}"
    )


PROJECT_ROOT = find_project_root()

PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

PRICE_FILE = (
    PROCESSED_DIR / "adjusted_close_prices.csv"
)

RETURN_FILE = (
    PROCESSED_DIR / "daily_returns.csv"
)

OUTPUT_FILE = (
    PROCESSED_DIR / "lstm_mse_features.csv"
)

OUTPUT_PARQUET = (
    PROCESSED_DIR / "lstm_mse_features.parquet"
)

In [3]:
MOMENTUM_HORIZONS = [
    1,
    21,
    63,
    126,
    252,
]

MACD_PAIRS = [
    (8, 24),
    (16, 48),
    (32, 96),
]

PRICE_STD_WINDOW = 63
MACD_STD_WINDOW = 252

VOLATILITY_HALFLIFE = 60
ANNUALIZATION_FACTOR = 252

In [5]:
prices = pd.read_csv(
    PRICE_FILE,
    index_col=0,
    parse_dates=True,
)

daily_returns = pd.read_csv(
    RETURN_FILE,
    index_col=0,
    parse_dates=True,
)


prices.index.name = "date"
daily_returns.index.name = "date"

prices.columns.name = "ticker"
daily_returns.columns.name = "ticker"


common_tickers = prices.columns.intersection(
    daily_returns.columns
)

prices = prices[common_tickers].copy()
daily_returns = daily_returns[common_tickers].copy()


prices, daily_returns = prices.align(
    daily_returns,
    join="inner",
    axis=0,
)


print("Prices shape :", prices.shape)
print("Returns shape:", daily_returns.shape)
print("Tickers      :", len(common_tickers))

display(prices.tail())

Prices shape : (11765, 20)
Returns shape: (11765, 20)
Tickers      : 20


ticker,AAPL,AMZN,BA,BAC,CAT,CVX,GE,GOOGL,GS,JNJ,JPM,KO,MCD,MSFT,NKE,PFE,PG,UNH,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,
2026-08-31,316.850006,259.769989,207.779999,61.625580,797.469971,206.139999,335.709991,339.132019,"1,020.900024",265.850006,356.019989,88.669998,261.680023,507.290009,38.650002,28.459999,145.119995,389.410004,104.870003,160.949997
2026-09-01,325.130005,254.919998,205.660004,61.675331,779.159973,211.050003,331.089996,334.804779,"1,002.559998",271.190002,354.950012,88.000000,261.109985,501.019989,38.119999,28.549999,146.210007,396.299988,105.919998,164.550003
2026-09-02,324.959991,254.979996,208.869995,62.282230,792.280029,211.779999,329.500000,336.903442,"1,004.419983",275.209991,356.220001,88.239998,260.950012,496.820007,38.240002,29.020000,147.639999,399.660004,106.089996,164.149994
2026-09-03,328.209991,258.899994,210.509995,62.719997,800.140015,211.320007,333.480011,342.260010,"1,037.930054",278.429993,362.059998,88.809998,259.630005,510.119995,38.770000,28.809999,146.919998,400.940002,108.419998,162.210007
2026-09-04,319.970001,258.510010,212.250000,62.680000,813.940002,208.600006,337.119995,338.459991,"1,038.609985",275.230011,358.640015,88.070000,255.690002,499.700012,38.400002,28.450001,146.440002,397.140015,107.139999,159.470001


In [8]:
assert not prices.empty
assert not daily_returns.empty

assert prices.index.equals(
    daily_returns.index
)

assert prices.columns.equals(
    daily_returns.columns
)

assert prices.index.is_monotonic_increasing
assert not prices.index.duplicated().any()

assert (
    prices.dropna() > 0
).all().all()

print("Raw datasets successfully validated.")

Raw datasets successfully validated.


# Volatility exponentielle : 
###σannual,t​=EWMAStd60​(rt​)252
​### σdaily,t​=​σannual,t​​ / sqrt(252)

In [9]:
daily_volatility = (
    daily_returns
    .ewm(
        halflife=VOLATILITY_HALFLIFE,
        adjust=False,
        min_periods=VOLATILITY_HALFLIFE,
        ignore_na=True,
    )
    .std(bias=False)
)

annualized_volatility = (
    daily_volatility
    * np.sqrt(ANNUALIZATION_FACTOR)
)

# Momentum in multiple horizons

In [10]:
momentum_features = {}

for horizon in MOMENTUM_HORIZONS:

    historical_return = prices.pct_change(
        periods=horizon,
        fill_method=None,
    )

    horizon_volatility = (
        daily_volatility
        * np.sqrt(horizon)
    )

    normalized_momentum = (
        historical_return
        / horizon_volatility
    )

    momentum_features[
        f"momentum_{horizon}d"
    ] = normalized_momentum

In [11]:
def compute_ewma(
    prices: pd.DataFrame,
    time_scale: int,
) -> pd.DataFrame:

    return prices.ewm(
        alpha=1 / time_scale,
        adjust=False,
        min_periods=time_scale,
        ignore_na=True,
    ).mean()

In [12]:
def compute_macd_trend_score(
    prices: pd.DataFrame,
    short_scale: int,
    long_scale: int,
    price_std_window: int = 63,
    signal_std_window: int = 252,
) -> pd.DataFrame:

    ewma_short = compute_ewma(
        prices,
        short_scale,
    )

    ewma_long = compute_ewma(
        prices,
        long_scale,
    )

    raw_macd = (
        ewma_short - ewma_long
    )

    price_std = (
        prices
        .rolling(
            window=price_std_window,
            min_periods=price_std_window,
        )
        .std(ddof=1)
        .replace(0, np.nan)
    )

    normalized_macd = (
        raw_macd / price_std
    )

    macd_std = (
        normalized_macd
        .rolling(
            window=signal_std_window,
            min_periods=signal_std_window,
        )
        .std(ddof=1)
        .replace(0, np.nan)
    )

    trend_score = (
        normalized_macd / macd_std
    )

    return trend_score

In [13]:
macd_features = {}

for short_scale, long_scale in MACD_PAIRS:

    feature_name = (
        f"macd_{short_scale}_{long_scale}"
    )

    macd_features[feature_name] = (
        compute_macd_trend_score(
            prices=prices,
            short_scale=short_scale,
            long_scale=long_scale,
            price_std_window=PRICE_STD_WINDOW,
            signal_std_window=MACD_STD_WINDOW,
        )
    )

In [14]:
daily_returns.shift(-1)

ticker,AAPL,AMZN,BA,BAC,CAT,CVX,GE,GOOGL,GS,JNJ,JPM,KO,MCD,MSFT,NKE,PFE,PG,UNH,WMT,XOM
date,,,,,,,,,,,,,,,,,,,,
1980-01-02,NaN,NaN,0.012594,-0.009259,-0.009390,0.002305,0.012820,NaN,NaN,0.001629,NaN,0.025927,-0.002924,NaN,NaN,0.000000,-0.013468,NaN,-0.011187,-0.025523
1980-01-03,NaN,NaN,0.097015,0.000000,0.014218,-0.004598,0.032912,NaN,NaN,0.014634,NaN,0.010831,0.014663,NaN,NaN,0.036666,0.005119,NaN,0.022629,0.009524
1980-01-04,NaN,NaN,0.036281,0.009346,-0.004673,0.000000,0.034313,NaN,NaN,-0.001603,NaN,-0.003572,-0.008670,NaN,NaN,0.000000,-0.005093,NaN,-0.003695,-0.004717
1980-01-07,NaN,NaN,0.015317,-0.009259,0.009389,0.009238,0.035545,NaN,NaN,0.040129,NaN,0.007168,0.014577,NaN,NaN,0.048232,0.003413,NaN,0.018523,0.007109
1980-01-08,NaN,NaN,-0.025862,0.046730,0.027907,-0.009153,-0.011442,NaN,NaN,-0.015432,NaN,0.007117,0.017241,NaN,NaN,-0.021473,0.013606,NaN,0.003643,-0.009411
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-31,0.026132,-0.018670,-0.010203,0.000807,-0.022960,0.023819,-0.013762,-0.012760,-0.017965,0.020087,-0.003005,-0.007556,-0.002178,-0.012360,-0.013713,0.003162,0.007511,0.017693,0.010012,0.022367
2026-09-01,-0.000523,0.000235,0.015608,0.009840,0.016839,0.003459,-0.004802,0.006268,0.001855,0.014824,0.003578,0.002727,-0.000613,-0.008383,0.003148,0.016462,0.009780,0.008478,0.001605,-0.002431
2026-09-02,0.010001,0.015374,0.007852,0.007029,0.009921,-0.002172,0.012079,0.015899,0.033363,0.011700,0.016394,0.006460,-0.005058,0.026770,0.013860,-0.007236,-0.004877,0.003203,0.021963,-0.011818


In [15]:
next_daily_return = (
    daily_returns.shift(-1)
)

normalized_target = (
    next_daily_return
    / daily_volatility
)

In [16]:
def wide_to_long(
    dataframe: pd.DataFrame,
    value_name: str,
) -> pd.DataFrame:

    return (
        dataframe
        .rename_axis(
            index="date",
            columns="ticker",
        )
        .stack()
        .rename(value_name)
        .reset_index()
    )

In [17]:
lstm_dataset = wide_to_long(
    prices,
    value_name="close",
)

In [19]:
base_variables = {
    "daily_return": daily_returns,
    "daily_volatility": daily_volatility,
    "annualized_volatility": annualized_volatility,
}

for variable_name, variable_matrix in base_variables.items():

    variable_long = wide_to_long(
        variable_matrix,
        value_name=variable_name,
    )

    lstm_dataset = lstm_dataset.merge(
        variable_long,
        on=["date", "ticker"],
        how="left",
    )

In [20]:
for feature_name, feature_matrix in momentum_features.items():

    feature_long = wide_to_long(
        feature_matrix,
        value_name=feature_name,
    )

    lstm_dataset = lstm_dataset.merge(
        feature_long,
        on=["date", "ticker"],
        how="left",
    )

In [21]:
for feature_name, feature_matrix in macd_features.items():

    feature_long = wide_to_long(
        feature_matrix,
        value_name=feature_name,
    )

    lstm_dataset = lstm_dataset.merge(
        feature_long,
        on=["date", "ticker"],
        how="left",
    )

In [22]:
target_long = wide_to_long(
    normalized_target,
    value_name="target_next_return",
)

lstm_dataset = lstm_dataset.merge(
    target_long,
    on=["date", "ticker"],
    how="left",
)

In [23]:
lstm_dataset = (
    lstm_dataset
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

In [24]:
FEATURE_COLUMNS = [
    "momentum_1d",
    "momentum_21d",
    "momentum_63d",
    "momentum_126d",
    "momentum_252d",
    "macd_8_24",
    "macd_16_48",
    "macd_32_96",
    "annualized_volatility",
]

TARGET_COLUMN = "target_next_return"

In [25]:
required_columns = (
    FEATURE_COLUMNS
    + [TARGET_COLUMN]
)

lstm_dataset_ready = (
    lstm_dataset
    .dropna(
        subset=required_columns
    )
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

In [26]:
print("Complete dataset:", lstm_dataset.shape)
print("Ready dataset   :", lstm_dataset_ready.shape)

display(
    lstm_dataset_ready[
        [
            "date",
            "ticker",
            *FEATURE_COLUMNS,
            TARGET_COLUMN,
        ]
    ].head()
)

Complete dataset: (235300, 15)
Ready dataset   : (209560, 15)


,date,ticker,momentum_1d,momentum_21d,momentum_63d,momentum_126d,momentum_252d,macd_8_24,macd_16_48,macd_32_96,annualized_volatility,target_next_return
0,1982-04-28,AAPL,-1.213029,-0.777078,-0.932321,-0.646492,-0.886314,-1.250734,-1.774675,-2.062475,0.536308,0.000000
1,1982-04-29,AAPL,0.000000,-0.866191,-1.025023,-0.712756,-0.908737,-1.347746,-1.836442,-2.091486,0.533249,0.254418
2,1982-04-30,AAPL,0.255707,-0.822220,-1.040703,-0.674827,-0.905039,-1.424241,-1.904536,-2.139254,0.530560,1.014194
3,1982-05-03,AAPL,1.012877,-0.918428,-0.911971,-0.632253,-0.866221,-1.435992,-1.951596,-2.189651,0.531250,0.980100
4,1982-05-04,AAPL,0.979397,-0.734150,-0.835988,-0.565272,-0.827634,-1.400477,-1.992372,-2.257853,0.531631,-0.474359


In [27]:
assert not lstm_dataset_ready.empty

assert not lstm_dataset_ready.duplicated(
    subset=["date", "ticker"]
).any()

assert lstm_dataset_ready[
    required_columns
].notna().all().all()

assert np.isfinite(
    lstm_dataset_ready[
        required_columns
    ].to_numpy()
).all()

assert TARGET_COLUMN not in FEATURE_COLUMNS

print("LSTM-MSE feature dataset successfully validated.")

LSTM-MSE feature dataset successfully validated.


In [29]:
ticker_test = "AAPL"

test_panel = (
    lstm_dataset[
        lstm_dataset["ticker"] == ticker_test
    ]
    .sort_values("date")
    .copy()
)

test_panel["observed_next_return"] = (
    test_panel["daily_return"].shift(-1)
)

test_panel["reconstructed_target"] = (
    test_panel["observed_next_return"]
    / test_panel["daily_volatility"]
)

valid_test = test_panel[
    [
        "target_next_return",
        "reconstructed_target",
    ]
].dropna()

assert np.allclose(
    valid_test["target_next_return"],
    valid_test["reconstructed_target"],
)

print("The target correctly represents the next-day return.")

The target correctly represents the next-day return.


In [30]:
dataset_report = (
    lstm_dataset_ready
    .groupby("ticker")
    .agg(
        first_ready_date=("date", "min"),
        last_ready_date=("date", "max"),
        observations=("date", "size"),
    )
)

display(dataset_report)

,first_ready_date,last_ready_date,observations
ticker,,,
AAPL,1982-04-28,2026-09-03,11178
AMZN,1998-09-29,2026-09-03,7026
BA,1981-05-15,2026-09-03,11418
BAC,1981-05-15,2026-09-03,11418
CAT,1981-05-15,2026-09-03,11418
CVX,1981-05-15,2026-09-03,11418
GE,1981-05-15,2026-09-03,11418
GOOGL,2006-01-03,2026-09-03,5200
GS,2000-09-14,2026-09-03,6531


In [31]:
lstm_dataset_ready.to_csv(
    OUTPUT_FILE,
    index=False,
)

lstm_dataset_ready.to_parquet(
    OUTPUT_PARQUET,
    index=False,
    engine="pyarrow",
)

print("Saved files:")
print(OUTPUT_FILE)
print(OUTPUT_PARQUET)

Saved files:
C:\Users\SETUP GAME\OneDrive\Desktop\Sun_project\Time_series_momentum\data\processed\lstm_mse_features.csv
C:\Users\SETUP GAME\OneDrive\Desktop\Sun_project\Time_series_momentum\data\processed\lstm_mse_features.parquet
